In [2]:
from pathlib import Path

path = Path("data/osm/california-260203.osm.pbf/").resolve()

print(path)

C:\Users\ddavi\Projects\scenic-route\processing\src\data\osm\california-260203.osm.pbf


### Runtimes
Sparse index + filters: 4m 27.3s  
Filters: 3m 53.8s  
Handlerv1 (355664 cells): 5m 5.0s  
Handlerv2 (355581 cells): 10m 35.6s  
Handlerv3 (336732 cells): 10m 33.4s
Handlerv3 run 2(355581 cells): 8m 20.5s  

In [ ]:
from classes.scenic_handler_v1 import ScenicHandler as ScenicHandlerV1
import json
import time
from osmium.filter import KeyFilter

start = time.time()

RESOLUTION = 8

water_data = json.load(open("constants/natural_water_tags.json"))
vegetation_data = json.load(open("constants/natural_vegetation_tags.json"))
geological_data = json.load(open("constants/natural_geological_tags.json"))
waterway_data = json.load(open("constants/waterway_tags.json"))

handler = ScenicHandlerV1(
    water_data, vegetation_data, geological_data, waterway_data, resolution=RESOLUTION
)
handler.apply_file(
    path,
    locations=True,
    idx="sparse_file_array",
    filters=[
        KeyFilter("natural", "landcover", "waterway", "tourism", "landuse", "leisure")
    ],
)

elapsed = time.time() - start
print(f"Took {elapsed:.2f}s")

print(f"Parsed {len(handler.cells)} H3 cells")

Water tags: ['bay', 'beach', 'blowhole', 'cape', 'coastline', 'crevasse', 'geyser', 'glacier', 'hot_spring', 'isthmus', 'mud', 'peninsula', 'reef', 'shingle', 'shoal', 'spring', 'strait', 'water', 'wetland']
Vegetation tags: ['fell', 'grassland', 'heath', 'moor', 'scrub', 'shrubbery', 'tree', 'tree_row', 'tundra', 'wood']
Geological tags: ['arch', 'arete', 'bare_rock', 'blockfield', 'cave_entrance', 'cliff', 'dune', 'earth_bank', 'fumarole', 'hill', 'peak', 'ridge', 'rock', 'saddle', 'sand', 'scree', 'sinkhole', 'stone', 'valley', 'volcano']
Waterway tags: ['river', 'riverbank', 'stream', 'tidal_channel', 'canal', 'ditch', 'fairway', 'dam', 'weir', 'waterfall']


### Runtime: 

In [81]:
import json

with open("data/output/scenic_cells.json", "w") as f:
    json.dump(handler.cells, f)

print(f"Saved {len(handler.cells)} H3 cells")

Saved 355581 H3 cells


In [82]:
import pandas as pd
import json
import os
import numpy as np

print(os.getcwd())

c:\Users\ddavi\Projects\scenic-route\processing\src


In [83]:
# with open("data/output/scenic_cells_v1.json", "r") as f:
#     cells = json.load(f)

cells = handler.cells

In [84]:
# Convert to DataFrame for easy scoring
df = pd.DataFrame.from_dict(cells, orient="index")
df.index.name = "h3_cell"
df.reset_index(inplace=True)

# Scenic score formula
df["diversity"] = (df[["water", "landcover", "relief", "recreation", "viewpoint"]] > 0).sum(axis=1)

feature_cols = ["water", "landcover", "relief", "recreation", "viewpoint", "urban"]
df[feature_cols] = df[feature_cols].apply(np.log1p)

df["raw_score"] = (
    4 * np.sqrt(df["water"])
    + 3 * np.sqrt(df["relief"])
    + 2 * np.sqrt(df["landcover"])
    + 2 * np.sqrt(df["recreation"])
    + 1.5 * df["viewpoint"]
    + 2 * np.log1p(df["diversity"])
    - 1 * np.sqrt(df["urban"])
)

# Normalize to 0–100
df["score"] = df["raw_score"].rank(pct=True) * 100

df_ranked = df.sort_values("score", ascending=False)
print(df_ranked[["h3_cell", "score"]].head(200))

               h3_cell       score
11546  8829a0b43dfffff  100.000000
259    8829a0b431fffff   99.999719
279    8829a0b435fffff   99.999438
10588  88291a4549fffff   99.999156
18994  8829a41003fffff   99.998875
...                ...         ...
29674  8828328d93fffff   99.945160
32143  88283099e9fffff   99.944879
20346  88283444a5fffff   99.944598
22129  882830275bfffff   99.944316
13536  882830885bfffff   99.944035

[200 rows x 2 columns]


In [85]:
from datetime import datetime
from pathlib import Path

output_dir = Path("data/output") / datetime.now().strftime("%Y%m%d")
output_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%H%M%S")

df_ranked.to_json(output_dir / f"scenic_scores_{ts}.json", orient="records")
df_ranked.to_csv(output_dir / f"scenic_scores_{ts}.csv", index=False)

In [86]:
# Detect skew
print("Raw score:")
print(df["raw_score"].describe())
print()
print(df["raw_score"].quantile([0.1, 0.33, 0.5, 0.75, 0.9, 0.95, 0.99]))

Raw score:
count    355581.000000
mean          8.493230
std           3.875767
min          -2.863758
25%           7.315510
50%           9.068877
75%          10.066875
max          34.973225
Name: raw_score, dtype: float64

0.10     3.482589
0.33     8.075213
0.50     9.068877
0.75    10.066875
0.90    12.492845
0.95    14.302120
0.99    19.107399
Name: raw_score, dtype: float64


In [87]:
print("\nFinal Score")
print(df["score"].describe())
print()
print(df["score"].quantile([0.1, 0.33, 0.5, 0.75, 0.9, 0.95, 0.99]))


Final Score
count    355581.000000
mean         50.000141
std          28.864992
min           0.000281
25%          25.048582
50%          50.153411
75%          75.007523
max         100.000000
Name: score, dtype: float64

0.10     9.849233
0.33    33.000188
0.50    50.153411
0.75    75.007523
0.90    90.000169
0.95    95.000014
0.99    99.000003
Name: score, dtype: float64


In [88]:
# find mammoth lakes cells
mammoth_lat, mammoth_lng = 37.6488, -118.9718
import h3

mammoth_cell = h3.latlng_to_cell(mammoth_lat, mammoth_lng, RESOLUTION)
print(df[df["h3_cell"] == mammoth_cell])

               h3_cell     water  landcover  relief  recreation  viewpoint  \
21465  88298cbad9fffff  4.343805    2.70805     0.0         0.0        0.0   

         urban  diversity  raw_score      score  
21465  4.70953          2  11.655029  87.697037  
